# 04 — Redes Neuronales por Cluster
**¿Cuándo se justifica una red neuronal?**

Una red neuronal se justifica si:
1. El tamaño del cluster es suficiente (n > 500 en train)
2. XGBoost/RF no logran AUC > 0.70 (señal de que la relación es altamente no lineal)
3. Existen interacciones de alto orden que los árboles no capturan eficientemente

Este notebook implementa un MLP (Multi-Layer Perceptron) ligero con sklearn y,
si PyTorch está disponible, una red con dropout y early stopping.

> **Nota:** En la mayoría de los clusters de este dataset el tamaño y la complejidad
> **no justifican** una red neuronal profunda. La comparación sirve para documentar
> ese hallazgo como experimento negativo metodológicamente válido.

## 0. Setup

In [1]:
import warnings; warnings.filterwarnings('ignore')
import numpy  as np
import pandas as pd
import matplotlib.pyplot  as plt
import matplotlib.patches as mpatches
from pathlib import Path
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.metrics import (roc_auc_score, recall_score, f1_score, precision_score,
                              classification_report, confusion_matrix,
                              roc_curve, precision_recall_curve)
from sklearn.impute   import SimpleImputer
from sklearn.preprocessing import StandardScaler
import time

# ── Rutas ─────────────────────────────────────────────────────────────────────
DATA_DIR = Path('../../data')
IMG_DIR  = Path('../../results/figures/clusters'); IMG_DIR.mkdir(parents=True, exist_ok=True)
CLUSTERED_CSV = DATA_DIR / 'dataset_clustered.csv'

assert CLUSTERED_CSV.exists(), (
    "No se encontró dataset_clustered.csv — ejecutar 00_cluster_profiles.ipynb primero"
)

df = pd.read_csv(CLUSTERED_CSV, low_memory=False)
print(f"✓ dataset_clustered.csv: {df.shape}")

# ── Constantes ────────────────────────────────────────────────────────────────
SEED         = 42
TARGET       = 'retention'
PRETEC21     = ['AD14','AD15','AD16','AD17','AD18']
TEC21        = ['AD19','AD20']
N_CLUSTERS   = df['cluster'].nunique()
MIN_AUC      = 0.60
SKF          = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# ── Features del modelo (excluir fuga y metadatos) ────────────────────────────
EXCLUDE = {TARGET, 'cluster', 'generation', 'regime', 'educational.model'}
FEATURE_COLS_BASE = [c for c in [
    'PNA','admission_test_norm','english.evaluation','admission.rubric','general.math.eval',
    'online.test','FTE','apoyo_financiero',
    'has_extracurriculars','has_physical','has_cultural','has_social',
    'first_gen_enc','educ_padres_max','parents_exatec_enc','socioec_enc','social_lag_enc',
    'age','is_male','estuvo_prepa_tec',
    'foreign_Yes: Foreigner','foreign_Yes: National',
    'zone_enc','zone_Rural','zone_Semiurban','zone_Urban',
    'school_enc','region_enc',
    'first_gen_present','parents_edu_present','took_admission_test',
    'has_socioeconomic_data','has_social_lag_data','has_zone_data',
] if c in df.columns and c not in EXCLUDE]

print(f"Features disponibles: {len(FEATURE_COLS_BASE)}")

# ── Paleta de clusters ────────────────────────────────────────────────────────
CLUSTER_COLORS = ['#2563eb','#dc2626','#16a34a','#f59e0b','#7c3aed','#0891b2']

# ── Helper: preparar datos de un cluster ──────────────────────────────────────
def get_cluster_data(cluster_id, regime=None):
    mask = df['cluster'] == cluster_id
    if regime == 'PreTec21':
        mask &= df['generation'].isin(PRETEC21)
    elif regime == 'Tec21':
        mask &= df['generation'].isin(TEC21)
    sub = df[mask].copy()
    feat_cols = [c for c in FEATURE_COLS_BASE if sub[c].std() > 0]
    imp = SimpleImputer(strategy='median')
    X   = imp.fit_transform(sub[feat_cols].values.astype(float))
    y   = sub[TARGET].values.astype(int)
    return X, y, feat_cols, sub

# ── Helper: evaluar modelo ────────────────────────────────────────────────────
def eval_model(model, X, y, model_name, feat_cols, seed=SEED):
    if len(np.unique(y)) < 2 or (y==0).sum() < 5:
        print(f"    ⚠ Insuficientes desertores — omitido")
        return None
    X_tr, X_te, y_tr, y_te = train_test_split(
        X, y, test_size=0.2, random_state=seed, stratify=y)
    model.fit(X_tr, y_tr)
    has_proba = hasattr(model, 'predict_proba')
    y_proba   = model.predict_proba(X_te)[:,1] if has_proba else None
    y_pred    = model.predict(X_te)
    auc  = roc_auc_score(y_te, y_proba) if (has_proba and len(np.unique(y_te))>1) else 0.5
    rec  = recall_score(y_te, y_pred, zero_division=0)
    prec = precision_score(y_te, y_pred, zero_division=0)
    f1   = f1_score(y_te, y_pred, zero_division=0)
    return dict(model=model_name, auc=auc, recall=rec, precision=prec, f1=f1,
                y_proba=y_proba, y_pred=y_pred, y_te=y_te, feat_cols=feat_cols,
                n_train=len(y_tr), n_test=len(y_te),
                dropout_rate=(y==0).mean(), trained_model=model)

print("\n✓ Setup completo — listo para entrenar modelos por cluster")

✓ dataset_clustered.csv: (77517, 39)
Features disponibles: 34

✓ Setup completo — listo para entrenar modelos por cluster


In [2]:
from sklearn.neural_network import MLPClassifier

try:
    import torch; import torch.nn as nn; TORCH_AVAILABLE = True
    print(f"✓ PyTorch {torch.__version__}")
except ImportError:
    TORCH_AVAILABLE = False
    print("⚠ PyTorch no disponible — usando sklearn MLP (ligero)")

try:
    import shap; SHAP_AVAILABLE = True
except ImportError:
    SHAP_AVAILABLE = False

⚠ PyTorch no disponible — usando sklearn MLP (ligero)


## 1. Análisis de justificación por cluster

In [3]:
# Determinar si la NN se justifica para cada cluster
print("═══ Análisis de justificación de Red Neuronal ═══")
print(f"{'Cluster':<10} {'Régimen':<12} {'n_train':<10} {'Justificado':<12} {'Razón'}")
print("─"*65)

justified = {}
for cluster_id in range(N_CLUSTERS):
    for regime_name in ['PreTec21','Tec21']:
        X, y, feat_cols, _ = get_cluster_data(cluster_id, regime_name)
        n = len(y)
        n_dropout = (y==0).sum()
        n_train   = int(n * 0.8)
        reason = []
        if n_train < 300:
            reason.append(f"n_train={n_train} < 300")
        if n_dropout < 20:
            reason.append(f"desertores={n_dropout} < 20")
        ok = len(reason) == 0
        justified[(cluster_id, regime_name)] = ok
        status = "✓ Sí" if ok else "✗ No"
        print(f"  C{cluster_id:<7}  {regime_name:<12}  {n_train:<10}  {status:<12}  "
              f"{', '.join(reason) if reason else 'tamaño suficiente'}")

═══ Análisis de justificación de Red Neuronal ═══
Cluster    Régimen      n_train    Justificado  Razón
─────────────────────────────────────────────────────────────────
  C0        PreTec21      16087       ✓ Sí          tamaño suficiente
  C0        Tec21         103         ✗ No          n_train=103 < 300
  C1        PreTec21      7693        ✓ Sí          tamaño suficiente


  C1        Tec21         15590       ✓ Sí          tamaño suficiente
  C2        PreTec21      2114        ✓ Sí          tamaño suficiente
  C2        Tec21         2503        ✓ Sí          tamaño suficiente
  C3        PreTec21      16512       ✓ Sí          tamaño suficiente
  C3        Tec21         1408        ✓ Sí          tamaño suficiente


## 2. MLP con sklearn (ligero, rápido)

In [4]:
results_mlp = {}

for cluster_id in range(N_CLUSTERS):
    for regime_name in ['PreTec21','Tec21']:
        key = (cluster_id, regime_name)
        if not justified.get(key, False):
            print(f"Cluster {cluster_id} [{regime_name}] — omitido (n insuficiente)")
            continue

        X, y, feat_cols, _ = get_cluster_data(cluster_id, regime_name)
        # Escalar (crítico para MLP)
        sc  = StandardScaler()
        X_s = sc.fit_transform(X)

        # Calcular class_weight manual (MLPClassifier no tiene class_weight)
        neg_w = len(y) / (2 * (y==0).sum())
        pos_w = len(y) / (2 * (y==1).sum())
        sample_weights_avail = False  # MLP no lo acepta directamente

        mlp = MLPClassifier(
            hidden_layer_sizes=(64, 32),
            activation='relu',
            solver='adam',
            alpha=1e-3,        # regularización L2
            learning_rate_init=1e-3,
            max_iter=300,
            early_stopping=True,
            validation_fraction=0.15,
            random_state=SEED,
            verbose=False
        )
        res = eval_model(mlp, X_s, y, 'MLP', feat_cols)
        if res is None: continue
        results_mlp[key] = res
        print(f"Cluster {cluster_id} [{regime_name}]  n={len(y):,}  "
              f"AUC={res['auc']:.3f}  Recall={res['recall']:.3f}  "
              f"{'✓' if res['auc']>=MIN_AUC else '⚠'}")

Cluster 0 [PreTec21]  n=20,109  AUC=0.569  Recall=1.000  ⚠
Cluster 0 [Tec21] — omitido (n insuficiente)
Cluster 1 [PreTec21]  n=9,617  AUC=0.497  Recall=1.000  ⚠


Cluster 1 [Tec21]  n=19,488  AUC=0.644  Recall=1.000  ✓
Cluster 2 [PreTec21]  n=2,643  AUC=0.510  Recall=1.000  ⚠
Cluster 2 [Tec21]  n=3,129  AUC=0.680  Recall=0.994  ✓


Cluster 3 [PreTec21]  n=20,641  AUC=0.665  Recall=0.995  ✓
Cluster 3 [Tec21]  n=1,761  AUC=0.559  Recall=0.997  ⚠


## 3. Red neuronal con PyTorch (con dropout y early stopping)

In [5]:
if TORCH_AVAILABLE:
    class DropoutNet(nn.Module):
        def __init__(self, n_features, hidden=(64, 32), dropout=0.3):
            super().__init__()
            layers = []
            prev = n_features
            for h in hidden:
                layers += [nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)]
                prev = h
            layers.append(nn.Linear(prev, 1))
            self.net = nn.Sequential(*layers)

        def forward(self, x):
            return torch.sigmoid(self.net(x))

    def train_pytorch_nn(X, y, hidden=(64,32), dropout=0.3, epochs=100,
                          lr=1e-3, patience=10, seed=SEED):
        torch.manual_seed(seed)
        X_tr, X_te, y_tr, y_te = train_test_split(
            X, y, test_size=0.2, random_state=seed, stratify=y)
        X_tr_t = torch.FloatTensor(X_tr); y_tr_t = torch.FloatTensor(y_tr).unsqueeze(1)
        X_te_t = torch.FloatTensor(X_te)
        pos_weight = torch.tensor([(y_tr==1).sum()/(y_tr==0).sum()], dtype=torch.float)
        criterion  = nn.BCELoss()
        model   = DropoutNet(X.shape[1], hidden, dropout)
        optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)

        best_val, best_ep, best_state = 0, 0, None
        for ep in range(epochs):
            model.train()
            optimizer.zero_grad()
            pred = model(X_tr_t)
            # weighted BCE manualmente
            weights = torch.where(y_tr_t == 0,
                                  torch.tensor(pos_weight.item()),
                                  torch.ones(y_tr_t.shape))
            loss = (criterion(pred, y_tr_t) * weights).mean()
            loss.backward(); optimizer.step()
            model.eval()
            with torch.no_grad():
                val_proba = model(X_te_t).numpy().flatten()
            val_auc = roc_auc_score(y_te, val_proba) if len(np.unique(y_te))>1 else 0.5
            if val_auc > best_val:
                best_val, best_ep = val_auc, ep
                best_state = {k: v.clone() for k, v in model.state_dict().items()}
            if ep - best_ep > patience: break
        model.load_state_dict(best_state)
        model.eval()
        with torch.no_grad():
            y_proba = model(X_te_t).numpy().flatten()
        y_pred = (y_proba >= 0.5).astype(int)
        return dict(auc=roc_auc_score(y_te, y_proba),
                    recall=recall_score(y_te, y_pred, zero_division=0),
                    f1=f1_score(y_te, y_pred, zero_division=0),
                    best_epoch=best_ep)

    results_nn = {}
    print("Entrenando redes neuronales PyTorch por cluster:")
    for cluster_id in range(N_CLUSTERS):
        for regime_name in ['PreTec21','Tec21']:
            key = (cluster_id, regime_name)
            if not justified.get(key, False): continue
            X, y, feat_cols, _ = get_cluster_data(cluster_id, regime_name)
            sc = StandardScaler(); X_s = sc.fit_transform(X)
            r  = train_pytorch_nn(X_s, y)
            results_nn[key] = r
            print(f"  C{cluster_id} [{regime_name}]  AUC={r['auc']:.3f}  "
                  f"Recall={r['recall']:.3f}  best_epoch={r['best_epoch']}")
else:
    results_nn = {}
    print("PyTorch no disponible — usar solo MLP de sklearn")

PyTorch no disponible — usar solo MLP de sklearn


## 4. Comparación final: ¿Vale la pena la NN?

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier

print("═══ Comparación: ¿Justifica la complejidad adicional? ═══")
print(f"{'Cluster':<10} {'Régimen':<12} {'LR':>7} {'RF':>7} {'MLP':>7} {'NN':>7} {'Ganancia MLP':>13}")
print("─"*65)

for cluster_id in range(N_CLUSTERS):
    for regime_name in ['PreTec21','Tec21']:
        key = (cluster_id, regime_name)
        if not justified.get(key, False): continue
        X, y, feat_cols, _ = get_cluster_data(cluster_id, regime_name)
        X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=SEED, stratify=y)
        sc = StandardScaler(); X_tr_s = sc.fit_transform(X_tr); X_te_s = sc.transform(X_te)

        lr = LogisticRegression(class_weight='balanced', max_iter=500, random_state=SEED)
        lr.fit(X_tr_s, y_tr)
        lr_auc = roc_auc_score(y_te, lr.predict_proba(X_te_s)[:,1]) if len(np.unique(y_te))>1 else 0.5

        rf = RandomForestClassifier(n_estimators=100, class_weight='balanced', random_state=SEED, n_jobs=-1)
        rf.fit(X_tr, y_tr)
        rf_auc = roc_auc_score(y_te, rf.predict_proba(X_te)[:,1]) if len(np.unique(y_te))>1 else 0.5

        mlp_auc = results_mlp.get(key, {}).get('auc', float('nan'))
        nn_auc  = results_nn.get(key, {}).get('auc', float('nan'))
        gain    = mlp_auc - lr_auc if not np.isnan(mlp_auc) else float('nan')
        verdict = "✓ Justifica" if gain > 0.02 else ("→ Marginal" if gain > 0 else "✗ No mejora")

        def fmt(v): return f'{v:.3f}' if not np.isnan(v) else '  N/A'
        print(f"  C{cluster_id:<7}  {regime_name:<12}  {fmt(lr_auc):>7}  {fmt(rf_auc):>7}  "
              f"{fmt(mlp_auc):>7}  {fmt(nn_auc):>7}  {verdict}")

print("\n→ Si la ganancia de MLP sobre LR es < 0.02, la complejidad NO se justifica")
print("  (principio de parsimonia: preferir el modelo más simple con poder equivalente)")

═══ Comparación: ¿Justifica la complejidad adicional? ═══
Cluster    Régimen           LR      RF     MLP      NN  Ganancia MLP
─────────────────────────────────────────────────────────────────


  C0        PreTec21        0.633    0.607    0.569      N/A  ✗ No mejora


  C1        PreTec21        0.671    0.637    0.497      N/A  ✗ No mejora


  C1        Tec21           0.655    0.612    0.644      N/A  ✗ No mejora
  C2        PreTec21        0.717    0.656    0.510      N/A  ✗ No mejora


  C2        Tec21           0.693    0.664    0.680      N/A  ✗ No mejora


  C3        PreTec21        0.665    0.653    0.665      N/A  → Marginal
  C3        Tec21           0.607    0.565    0.559      N/A  ✗ No mejora

→ Si la ganancia de MLP sobre LR es < 0.02, la complejidad NO se justifica
  (principio de parsimonia: preferir el modelo más simple con poder equivalente)
